**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/07-stereo-calibration.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/stereo-calibration) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6G · Stereo Calibration & Rectification

_Lecture 6, Section 6: stereo calibration._

Two cameras on one rig: calibrate each, find the rotation **R** and translation **T** between them, and **rectify** the
pair so that matching points lie on the same image row, which is the starting point for stereo depth (Lecture 10).
We use OpenCV's classic `left01…14 / right01…14` sample pairs (9 × 6 board). Two Raspberry Pi cameras on a bracket
work exactly the same way.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)

import lab6_checks as C
C.fetch_set('stereo')
print("data ready")

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C
np.set_printoptions(precision=4, suppress=True)

L = sorted(glob.glob("data/stereo/left*.jpg")); R = sorted(glob.glob("data/stereo/right*.jpg"))
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(cv2.imread(L[0], 0), cmap="gray"); ax[0].set_title("left01"); ax[1].imshow(cv2.imread(R[0], 0), cmap="gray"); ax[1].set_title("right01")
plt.show()

## 1. Corners found in **both** images of a pair

In [ ]:
pattern = (9, 6)
objp = C.board_points(*pattern)            # units: squares (multiply by the square size for mm)
objpoints, imgL, imgR = [], [], []
for a, b in zip(L, R):
    ga, gb = cv2.imread(a, 0), cv2.imread(b, 0)
    oka, ca = C.find_corners(ga, pattern); okb, cb = C.find_corners(gb, pattern)
    # TODO: keep the pair only if the board was found in BOTH images
    pass
size = ga.shape[::-1]
print(len(objpoints), "pairs, image size", size)

## 2. Calibrate each camera, then the pair

In [ ]:
rmsL, K1, D1, _, _ = cv2.calibrateCamera(objpoints, imgL, size, None, None)
rmsR, K2, D2, _, _ = cv2.calibrateCamera(objpoints, imgR, size, None, None)
print(f"left RMS {rmsL:.3f}, right RMS {rmsR:.3f}")

# TODO: cv2.stereoCalibrate with flags=cv2.CALIB_FIX_INTRINSIC
rms, R_, T, E, F = 99.0, np.eye(3), np.zeros(3), None, None

print("stereo RMS", rms); print("R =\n", R_); print("T =", np.ravel(T), " |T| =", np.linalg.norm(T), "squares")

In [ ]:
C.check_stereo(rms, T, R_)

<details><summary><b>Show a solution</b></summary>

```python
for a, b in zip(L, R):
    ...
    if oka and okb:
        objpoints.append(objp); imgL.append(ca); imgR.append(cb)

rms, K1, D1, K2, D2, R_, T, E, F = cv2.stereoCalibrate(
    objpoints, imgL, imgR, K1, D1, K2, D2, size, flags=cv2.CALIB_FIX_INTRINSIC)
```
</details>

**Check the epipolar constraint.** For matching points, $x_R^\top F x_L = 0$:

In [ ]:
xl = np.hstack([imgL[0].reshape(-1, 2), np.ones((54, 1))]); xr = np.hstack([imgR[0].reshape(-1, 2), np.ones((54, 1))])
print("mean |x_R^T F x_L| =", np.abs(np.sum(xr * (F @ xl.T).T, axis=1)).mean())

## 3. Rectify

In [ ]:
R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(K1, D1, K2, D2, size, R_, T, alpha=0)
mapL = cv2.initUndistortRectifyMap(K1, D1, R1, P1, size, cv2.CV_16SC2)
mapR = cv2.initUndistortRectifyMap(K2, D2, R2, P2, size, cv2.CV_16SC2)

i = 0
la, lb = cv2.imread(L[i]), cv2.imread(R[i])
ra, rb = cv2.remap(la, *mapL, cv2.INTER_LINEAR), cv2.remap(lb, *mapR, cv2.INTER_LINEAR)
def lines(a, b):
    p = np.hstack([a, b])
    for y in range(20, p.shape[0], 30): cv2.line(p, (0, y), (p.shape[1], y), (0, 255, 0), 1)
    return p
fig, ax = plt.subplots(2, 1, figsize=(14, 11))
ax[0].imshow(lines(la, lb)[:, :, ::-1]); ax[0].set_title("raw"); ax[1].imshow(lines(ra, rb)[:, :, ::-1]); ax[1].set_title("rectified")
for a in ax: a.axis("off")
plt.show()

**Measure it.** Find the corners in the rectified pair and compare their rows.

In [ ]:
oka, ca = C.find_corners(cv2.cvtColor(ra, cv2.COLOR_BGR2GRAY), pattern)
okb, cb = C.find_corners(cv2.cvtColor(rb, cv2.COLOR_BGR2GRAY), pattern)
print("rows left :", ca.reshape(-1, 2)[:5, 1].round(1)); print("rows right:", cb.reshape(-1, 2)[:5, 1].round(1))

In [ ]:
C.check_rectified(ca.reshape(-1, 2)[:, 1], cb.reshape(-1, 2)[:, 1])

## 4. A first disparity map (preview of Lecture 10)

After rectification, the same point is `d = u_left − u_right` pixels apart; depth is $Z = fB/d$.

In [ ]:
gl, gr = cv2.cvtColor(ra, cv2.COLOR_BGR2GRAY), cv2.cvtColor(rb, cv2.COLOR_BGR2GRAY)
sgbm = cv2.StereoSGBM_create(minDisparity=0, numDisparities=64, blockSize=7, P1=8 * 49, P2=32 * 49, uniquenessRatio=10)
disp = sgbm.compute(gl, gr).astype(np.float32) / 16
d_corners = ca.reshape(-1, 2)[:, 0] - cb.reshape(-1, 2)[:, 0]
print("disparity of the board corners: %.1f … %.1f px" % (d_corners.min(), d_corners.max()))
print("depth of the board ≈ f·B/d = %.1f squares" % (P1[0, 0] * np.linalg.norm(T) / d_corners.mean()))
plt.figure(figsize=(9, 6)); plt.imshow(disp, cmap="turbo", vmin=0, vmax=64); plt.colorbar(label="disparity (px)"); plt.axis("off"); plt.show()

## Your turn

1. The squares of this board are about 25 mm (assume so). What is the baseline in mm? How far is the board in pair 0?
2. Run `stereoCalibrate` **without** `CALIB_FIX_INTRINSIC` (it then refines K1, D1, K2, D2 too). Compare RMS and |T|.
3. Two Pi cameras: describe how you would capture **synchronised** pairs with Picamera2 (two `Picamera2(0)` /
   `Picamera2(1)` objects) and why a moving board needs hardware sync (or a still board).